#### By: Peyman Shahidi
#### Created: Aug 1, 2025

<br>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

main_folder_path = ".."
output_plot_path = f"{main_folder_path}/writeup/plots"
os.makedirs(output_plot_path, exist_ok=True)

# Short-run (time-only) parameters for a single fixed job: no skill, no hand-off.
# The job has two steps. Step 1 is short but hard for AI; step 2 is time-consuming
# but easy for AI. The firm minimizes total completion time (short-run problem).
t1m, t2m = 1, 4   # manual time costs
t1a, t2a = 1, 1   # AI management time per attempt
d1, d2 = 6, 1     # AI difficulty: q_i = alpha**d_i

alpha = np.linspace(0.001, 1, 4000)

both_manual = (t1m + t2m) * np.ones_like(alpha)
s1m_s2aug   = t1m + t2a / alpha**d2          # 1 + 1/alpha
s1aug_s2m   = t1a / alpha**d1 + t2m
both_aug    = t1a / alpha**d1 + t2a / alpha**d2
chain       = t2a / alpha**(d1 + d2)         # alpha**-7 : steps 1-2 as one AI chain

costs = np.vstack([both_manual, s1m_s2aug, s1aug_s2m, both_aug, chain])
min_cost = costs.min(axis=0)

# Frontier thresholds
bp1 = alpha[np.argmin(np.abs(both_manual - s1m_s2aug))]   # ~0.25
bp2 = alpha[np.argmin(np.abs(s1m_s2aug - chain))]         # ~0.90

# ---------- Panel (a): total time cost by AI strategy ----------
plt.figure(figsize=(8, 6))
plt.plot(alpha, both_manual, label="Both steps manual", alpha=0.55, color="C0")
plt.plot(alpha, s1m_s2aug, label="Step 1 manual, Step 2 augmented", alpha=0.55, color="C1")
plt.plot(alpha, both_aug, label="Both steps augmented", alpha=0.55, color="C3")
plt.plot(alpha, chain, label="Steps 1--2 chained (1 automated, 2 augmented)", alpha=0.55, color="C4")
plt.plot(alpha, min_cost, label="Minimum cost (optimal AI strategy)", color="black", linewidth=4)
for bp in (bp1, bp2):
    plt.axvline(bp, color="gray", linestyle="--", linewidth=1, alpha=0.8)
    plt.text(bp, 0.25, rf"$\alpha={bp:.2f}$", ha="center", va="bottom", fontsize=10, backgroundcolor="white")
plt.xlabel(r"AI Quality Level $\alpha$", fontsize=15)
plt.ylabel("Total Execution Time", fontsize=15)
plt.ylim(0, 12)
plt.xlim(0.1, 1)
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_plot_path}/example5_costs.png", dpi=300)
plt.show()

# ---------- Panel (b): marginal benefit of improving alpha ----------
mb_aug = t2a * d2 / alpha**(d2 + 1)               # 1/alpha^2
mb_chain = t2a * (d1 + d2) / alpha**(d1 + d2 + 1)  # 7/alpha^8
mb_opt = np.piecewise(
    alpha,
    [alpha <= bp1, (alpha > bp1) & (alpha <= bp2), alpha > bp2],
    [0, lambda a: t2a * d2 / a**(d2 + 1), lambda a: t2a * (d1 + d2) / a**(d1 + d2 + 1)],
)
plt.figure(figsize=(8, 6))
plt.plot(alpha, np.zeros_like(alpha), label="Both steps manual", alpha=0.55, color="C0")
plt.plot(alpha, mb_aug, label="Step 1 manual, Step 2 augmented", alpha=0.55, color="C1")
plt.plot(alpha, mb_chain, label="Steps 1--2 chained", alpha=0.55, color="C4")
plt.plot(alpha, mb_opt, label="Optimal AI strategy", color="black", linewidth=4)
for bp in (bp1, bp2):
    plt.axvline(bp, color="gray", linestyle="--", linewidth=1, alpha=0.8)
    plt.text(bp, 0.5, rf"$\alpha={bp:.2f}$", ha="center", va="bottom", fontsize=10, backgroundcolor="white")
plt.xlabel(r"AI Quality Level $\alpha$", fontsize=15)
plt.ylabel(r"Marginal Benefit of Increasing $\alpha$", fontsize=15)
plt.ylim(0, 25)
plt.xlim(0.1, 1)
plt.legend(loc="upper left")
plt.tight_layout()
plt.savefig(f"{output_plot_path}/example5_marginalBenefit.png", dpi=300)
plt.show()
